# Sheet 07 - Flow Matching and Diffusion

Introduction to Deep Learning - Summer Semester 2026

Ulf Krumnack & Robin Rawiel - Universität OsnabrückDue: June 7, 2026

In this exercise sheet you will connect the diffusion chapter to a
compact implementation. The lecture and script frame generation as
*reversing the arrow of time*: a fixed forward process slowly destroys
an image with Gaussian noise, and a neural network learns to run that
film backward. You will first work through the theory (forward and
reverse processes, the score function, the probability-flow ODE and its
link to flows and optimal transport, and how latent diffusion, CLIP and
guidance scale these ideas up). You will then implement a small
**denoising diffusion model (DDPM)** on MNIST, and finally explore a
deterministic, few-step sampler that makes the connection between
diffusion and continuous flows concrete.

## Task 1: Theory Questions \[6 points\]

### 1.1 Forward and Reverse Diffusion \[2 points\]

1.  The script motivates diffusion through the physical *arrow of time*.
    Explain the analogy: what plays the role of the “drop of ink”, and
    why is the forward (noising) process easy while the reverse
    (denoising) process is hard? What exactly does the neural network
    learn so that machine learning can do what physics cannot?

    <span style="color: green;"><b>Answer:</b></span> The "drop of ink" is a clean image $x_0$. **Forward (noising) is easy** because we just add Gaussian noise step by step — a fixed recipe, no learning. **Reverse (denoising) is hard** because many clean images could explain the same noisy one, so exact reversal would need the full data distribution. The network learns to **predict the noise** $\epsilon_\theta(x_t, t)$ in a noisy image; subtracting it step by step lets ML do the time-reversal that physics cannot.

    <details>
    <summary><b>More details</b></summary>

    - **Why physics can't reverse it**: the forward process throws away information randomly, so it's lost. A network trained on many images instead learns *which clean images are likely*, and uses that to make a good statistical guess.
    - **Example**: a noisy blob could be a "5" or a "6". Having seen thousands of each, the model nudges the pixels toward a plausible digit shape rather than random static.

    </details>

2.  Write down the single forward step $q(x_t \mid x_{t-1})$ as a
    Gaussian, and the closed-form marginal $q(x_t \mid x_0)$. Define the
    noise schedule $\{\beta_t\}$ and the quantity $\bar\alpha_t$.

    <span style="color: green;"><b>Answer:</b></span>

    **Single step:** $\;q(x_t \mid x_{t-1}) = \mathcal{N}\bigl(x_t;\;\underbrace{\sqrt{1-\beta_t}\,x_{t-1}}_{\text{mean}},\;\underbrace{\beta_t I}_{\text{variance}}\bigr)$

    - $x_t$ = noisy image at step $t$ (the variable whose distribution we describe).
    - $x_{t-1}$ = image from the previous step (what we noise).
    - $\sqrt{1-\beta_t}\,x_{t-1}$ = the **mean**: the old image slightly **shrunk** so values don't grow.
    - $\beta_t I$ = the **variance**: a small amount of fresh Gaussian noise added in every pixel ($I$ = identity, so each pixel is independent).

    **Marginal / shortcut:** $\;q(x_t \mid x_0) = \mathcal{N}\bigl(x_t;\;\underbrace{\sqrt{\bar\alpha_t}\,x_0}_{\text{mean}},\;\underbrace{(1-\bar\alpha_t) I}_{\text{variance}}\bigr)\;\Leftrightarrow\; x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$

    - $x_0$ = the clean starting image.
    - $\sqrt{\bar\alpha_t}\,x_0$ = how much of the **original image** is left at step $t$ (the signal).
    - $(1-\bar\alpha_t)I$ = how much **noise** has accumulated by step $t$. Signal + noise weights always sum to 1.
    - $\epsilon \sim \mathcal{N}(0,I)$ = one sample of standard noise; lets us jump to any $t$ in **one** step instead of looping.

    **The schedule terms:**

    - $\{\beta_t\}_{t=1}^{T}$ = **noise schedule**: how much noise is added at each step (small, slowly increasing, e.g. $10^{-4}\to0.02$).
    - $\alpha_t = 1-\beta_t$ = fraction of signal **kept** in one step.
    - $\bar\alpha_t = \prod_{s=1}^{t}\alpha_s$ = product of all those, i.e. the fraction of the original image that **survives** after $t$ steps.

    <details>
    <summary><b>More details</b></summary>

    - **Why shrink by $\sqrt{1-\beta_t}$**: without it, adding noise every step would make pixel values grow without bound. Shrinking keeps total variance roughly constant.
    - **Why the shortcut works**: summing many independent Gaussian noises gives another Gaussian, so chaining $t$ single steps collapses into one Gaussian with mean $\sqrt{\bar\alpha_t}\,x_0$ and variance $(1-\bar\alpha_t)I$.
    - **Edge cases**: at $t=0$, $\bar\alpha_0=1 \Rightarrow x_0$ untouched; at large $t$, $\bar\alpha_t\to0 \Rightarrow x_t\approx$ pure noise.

    </details>

3.  What is the **score function** of a distribution, and how is
    predicting the added noise $\epsilon$ related to estimating the
    score $\nabla_{x_t} \log q(x_t)$?

    <span style="color: green;"><b>Answer:</b></span> The **score function** is $\nabla_x \log q(x)$ — a vector field pointing "uphill" toward more realistic images on the probability landscape. The lecture notes the score is **proportional to the negative of the added noise**, so a network that predicts the noise $\epsilon_\theta$ is automatically estimating the score:
    $$\nabla_{x_t}\log q(x_t) \;\propto\; -\,\epsilon_\theta(x_t,t).$$
    Predicting noise and estimating the score are therefore the same task.

    <details>
    <summary><b>More details</b></summary>

    - **Picture**: peaks = realistic data, valleys = noise. Adding noise pushes the image *downhill*; the score points *uphill*. So "subtract the noise" = "step uphill" = denoise.
    - **Exact link** (Gaussian marginal): $\nabla_{x_t}\log q(x_t\mid x_0) = -\dfrac{\epsilon}{\sqrt{1-\bar\alpha_t}}$, i.e. the score is just the noise scaled by a known constant.
    - **Why it matters**: every reverse sampler (DDPM, reverse SDE, probability-flow ODE) needs the score. Once we have $\epsilon_\theta$, we can plug it into any of them.

    </details>

### 1.2 Flow View, Probability-Flow ODE and Optimal Transport \[2 points\]

1.  The forward process is a stochastic differential equation (SDE). The
    script states that every such SDE has a deterministic counterpart,
    the **probability-flow ODE**. What does this ODE describe, and why
    does it connect diffusion models to **normalizing flows**? What
    practical capability (used later for editing) does the
    deterministic, invertible mapping enable?

    <span style="color: green;"><b>Answer:</b></span> The **probability-flow ODE** describes the *same* spreading of the probability distribution as the random SDE, but with **no randomness** — one smooth, deterministic path from noise to data:

    $$dx_t = \Bigl[\,\underbrace{f(x_t,t)}_{\text{drift}} \;-\; \underbrace{\tfrac{1}{2}\,g(t)^2}_{\text{half noise strength}}\,\underbrace{\nabla_{x_t}\log p(x_t)}_{\text{score (uphill)}}\Bigr]\,dt$$

    - $f(x_t,t)$ = **drift**: the deterministic part that shrinks/scales pixels.
    - $g(t)$ = **diffusion coefficient**: noise strength in the original SDE (here only its square enters, no random term).
    - $\nabla_{x_t}\log p(x_t)$ = **score**: points uphill toward realistic data (estimated by $\epsilon_\theta$).
    - $\tfrac12$ = the factor that lets the deterministic flow mimic the spreading the random noise used to do.

    It connects to **normalizing flows** because, with no randomness, each noise point maps to **one unique** image along an **invertible** curve — exactly what a normalizing flow is (a bijective noise↔data map). Practically, this invertibility enables **deterministic inversion / image editing**: encode a real image to its noise, change the prompt, decode back, and the layout is preserved.

    <details>
    <summary><b>More details</b></summary>

    - **SDE vs ODE**: the reverse SDE keeps a random Wiener term $d\bar w$, so running it twice gives two different images. The ODE drops that term, so the same noise always gives the same image.
    - **Why the $\tfrac12$**: removing the random jitter loses the "spreading" effect of noise; halving the score term replaces it deterministically, so the distribution at each $t$ stays identical to the SDE's.
    - **Editing link (later in the script)**: deterministic inversion underlies real-photo editing — trace image→noise, swap the text prompt, trace noise→image, keeping structure intact.

    </details>

2.  From the **flow** viewpoint, the model follows a deterministic
    trajectory governed by an ODE. Conceptually, what is the network
    predicting in such a flow-based model, and how does this differ from
    the noise prediction used in DDPM? Why does generation reduce to
    solving an ODE?

    <span style="color: green;"><b>Answer:</b></span> In a flow model we define a **straight path** from data to noise and the network predicts the **velocity** (direction of travel) along it:

    $$x_t = \underbrace{(1-t)\,x_0}_{\text{data part}} + \underbrace{t\,\epsilon}_{\text{noise part}}, \qquad v = \frac{dx_t}{dt} = \underbrace{\epsilon - x_0}_{\text{constant velocity}}$$

    - $t$ = continuous time from $0$ (clean data) to $1$ (pure noise).
    - $x_t$ = point on the straight line between an image $x_0$ and noise $\epsilon$.
    - $v = \epsilon - x_0$ = **velocity**: constant, because a straight line has constant slope. The network $v_\theta(x_t,t)$ learns this vector.

    **Difference from DDPM:** DDPM predicts the **noise** $\epsilon_\theta$; a flow model predicts the **velocity** $v_\theta$ (the arrow telling you which way to move). **Generation = solving an ODE** because you start at noise and just follow the predicted velocity field step by step ($x_{t-\Delta} = x_t - v_\theta\,\Delta t$) until you reach the data — integrating $\tfrac{dx}{dt}=v_\theta$ is exactly solving an ODE.

    <details>
    <summary><b>More details</b></summary>

    - **Same idea, different target**: noise prediction and velocity prediction both define the trajectory; velocity is just a more direct "where do I go next" signal for straight paths.
    - **Why an ODE and not random steps**: the flow is deterministic (no injected noise), so the trajectory is fully fixed once you know the velocity field — numerically you integrate it like any ODE.

    </details>

3.  What problem do **optimal transport** and **Schrödinger bridges**
    address in this context, and why does straightening the trajectories
    let modern models sample in far fewer steps?

    <span style="color: green;"><b>Answer:</b></span> If we pair noise and images **randomly**, the straight lines connecting them **cross**. At a crossing the network gets **conflicting velocity targets** and outputs blurry averages. **Optimal transport** fixes this by pairing each noise vector with its **closest** image, so the paths become **straight and parallel (non-crossing)**. **Schrödinger bridges** do the same for *stochastic* paths — the most efficient noisy trajectories with a controlled amount of jitter. Straight paths have **constant velocity and no curvature**, so you can take a few **big** steps without falling off the curve — cutting sampling from ~1000 steps down to as few as 1–4.

    <details>
    <summary><b>More details</b></summary>

    - **Why curvature forces many steps**: a curved path means the direction keeps changing, so a big jump overshoots; you must take many small steps to stay on it.
    - **OT = "Earth Mover"**: cheapest way to move the pile of "noise" onto the pile of "data", i.e. minimal total distance — which naturally untangles the paths.
    - **Schrödinger bridge**: keeps some randomness (good for sharp details via Langevin-style jitter) but still takes the most direct route, avoiding wasteful wandering.
    - These ideas power recent models like Stable Diffusion 3 and Flux.

    </details>

### 1.3 Latent Diffusion, CLIP and Classifier-Free Guidance \[2 points\]

1.  Running a U-Net over millions of pixels for many steps is expensive.
    Explain how a **latent diffusion model** (e.g. Stable Diffusion)
    reduces this cost. Which component is trained separately and then
    frozen, and where does the diffusion process actually run?

    <span style="color: green;"><b>Answer:</b></span> A **latent diffusion model (LDM)** moves the whole diffusion process out of pixel space and into a much smaller **latent space**. A **VAE** is trained separately and then **frozen**: its encoder $E$ squashes an image into a compact latent $z_0 = E(x_0)$, and its decoder $D$ turns a latent back into pixels. Diffusion (adding noise + the U-Net denoising) runs **entirely on the small latents $z_t$**, not on raw pixels. After generating a clean latent $z_0$, the frozen decoder produces the final image $\hat{x}_0 = D(z_0)$.

    - $E$ = encoder (image → latent), **frozen** after separate training.
    - $z_0 = E(x_0)$ = compressed representation (e.g. 8× smaller per side).
    - $D$ = decoder (latent → pixels), also frozen.
    - The U-Net $\epsilon_\theta(z_t, t)$ works on latents, so each step processes far fewer numbers → much faster, less memory.

    <details>
    <summary><b>More details</b></summary>

    - **Why it's cheaper**: a 1024² image has millions of pixels; the latent grid is often 8× smaller per side ≈ 64× fewer values, so every denoising step is dramatically lighter.
    - **Why a VAE is safe to compress with**: images have lots of redundant high-frequency detail the eye ignores; the VAE keeps the *semantic* content and lets the decoder re-render fine pixels.
    - **Same objective**: the training loss is identical to standard diffusion, just computed on $z_0$ instead of $x_0$.

    </details>

2.  What does **CLIP** learn, and how does its contrastive training make
    it useful as a way to condition image generation on text?

    <span style="color: green;"><b>Answer:</b></span> **CLIP** learns a shared **joint embedding space** where an image and its caption land at the **same point**. It trains two encoders — image $E_{\text{img}}$ and text $E_{\text{txt}}$ — with a **contrastive** objective: in a batch, make matching image–text pairs have **high** similarity and mismatched pairs **low** similarity.

    $$S_{i,j} = v_i \cdot u_j, \qquad v_i = E_{\text{img}}(x_i),\; u_j = E_{\text{txt}}(c_j)$$

    - $v_i, u_j$ = image / text vectors in the joint space.
    - $S_{i,j}$ = similarity (dot product); the **diagonal** $S_{i,i}$ = correct pairs (pushed up), **off-diagonal** = wrong pairs (pushed down).

    This makes CLIP useful for **text conditioning**: pass a prompt through the frozen $E_{\text{txt}}$ to get a **conditioning vector** $c$ that lives in the same space as image features, so the U-Net can use it (via cross-attention) to steer generation toward the text.

    <details>
    <summary><b>More details</b></summary>

    - **Loss**: softmax + cross-entropy over each row (image picks its caption) and each column (caption picks its image), averaged; temperature $\tau$ sharpens it.
    - **Why contrastive (not captioning)**: matching is far easier to learn than generating exact words, yet still forces a deep, grounded alignment of language and vision.
    - **Hard negatives / big batches**: huge batches include subtly different captions ("dog running" vs "dog sitting"), forcing genuine understanding instead of shortcuts like background color.

    </details>

3.  Describe **classifier-free guidance**: how is the model trained to
    allow it, and what is the inference-time update? What is the
    practical tradeoff controlled by the guidance weight $w$?

    <span style="color: green;"><b>Answer:</b></span> **Classifier-free guidance (CFG)** makes the model follow the prompt more strongly. **Training:** randomly **drop the text** ~10–20% of the time, replacing it with an empty **null prompt** $\varnothing$, so the *same* U-Net learns both conditional ($c$) and unconditional ($\varnothing$) prediction. **Inference:** run the U-Net **twice** and combine the two predictions:

    $$\tilde\epsilon = \underbrace{\epsilon_\theta(x_t,t,\varnothing)}_{\text{unconditional}} + \;w\cdot\bigl(\underbrace{\epsilon_\theta(x_t,t,c)}_{\text{with text}} - \underbrace{\epsilon_\theta(x_t,t,\varnothing)}_{\text{without text}}\bigr)$$

    - $\epsilon_\theta(x_t,t,c)$ = prediction **with** the prompt.
    - $\epsilon_\theta(x_t,t,\varnothing)$ = prediction **ignoring** the prompt.
    - their difference = the part that is **specifically due to the text**.
    - $w$ = **guidance weight**: how much to amplify that text-specific direction ($w=1$ → plain conditional; $w\approx4$–$7$ → strong steering).

    **Tradeoff:** higher $w$ = stronger prompt adherence but **less diversity** and possible artifacts/oversaturation; lower $w$ = more varied/natural but looser prompt-following.

    <details>
    <summary><b>More details</b></summary>

    - **"Classifier-free"**: earlier methods needed a separate image classifier to push toward a class; CFG gets the same steering from the model's own conditional vs unconditional predictions — no extra classifier.
    - **Landscape view**: the difference term steepens the probability landscape toward the prompt's peak, away from generic-image valleys.
    - **Note**: CFG changes the vector field, which breaks exact ODE invertibility — relevant for editing (handled by null-text inversion).

    </details>

## Common Setup

We reuse the lightweight MNIST setup from the previous generative-model
sheets and keep the runtime manageable by working on subsets rather than
the full dataset. Images are mapped to $[-1, 1]$, which matches the
symmetric Gaussian noise used by the diffusion process.

In this part you should:

- load the MNIST subsets and create the data loaders,
- map images to $[-1, 1]$ via a `Normalize` transform, and
- add one helper for displaying image grids.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Map images from [0, 1] to [-1, 1] so they match the symmetric Gaussian noise.
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

full_train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
full_test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_subset = Subset(full_train_dataset, range(8000))
test_subset = Subset(full_test_dataset, range(1000))

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)


def show_grid(images, title, nrow=8):
    """Display a batch of [-1, 1] images as a grid."""
    # TODO: Your solution here

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:05<00:00, 1.88MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 224kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 1.88MB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 4.33MB/s]


In [ ]:
# TODO: Your solution here

## Task 2: Denoising Diffusion on MNIST \[10 points\]

**Learning objectives:**

- implement the forward diffusion process in closed form,
- build a small noise-prediction network with a time embedding,
- train it with the denoising (noise-prediction) objective, and
- sample new digits with the stochastic reverse process.

Tip: get the closed-form forward process right first and visualize it.
If sampling produces only static, double-check the indexing of the
schedule tensors and that the model is conditioned on the timestep $t$.

### 2.1 Forward Diffusion Process \[2 points\]

Implement the noise schedule and the closed-form forward process
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I),$$
then visualize how a single image is destroyed as $t$ increases.

In [ ]:
T = 1000


def linear_schedule(num_steps, beta_start=1e-4, beta_end=0.02):
    """Linearly increasing variance schedule {beta_t}."""
    # TODO: Your solution here


def compute_alpha_bar(betas):
    """Cumulative product alpha_bar_t = prod_{s<=t} (1 - beta_s)."""
    # TODO: Your solution here


betas = linear_schedule(T).to(device)
alphas = (1.0 - betas).to(device)
alpha_bar = compute_alpha_bar(betas).to(device)


def forward_diffusion(x_0, t, alpha_bar):
    """Sample x_t from q(x_t | x_0). Returns (x_t, noise)."""
    # Hint: sample Gaussian noise with the same shape as x_0 and reshape alpha_bar[t]
    # to (batch, 1, 1, 1) before applying the closed-form formula for x_t.
    # TODO: Your solution here

In [ ]:
# TODO: Your solution here

### 2.2 Noise-Prediction Network \[3 points\]

The denoiser $\epsilon_\theta(x_t, t)$ must know *how noisy* its input
is, so the timestep $t$ is encoded with a sinusoidal embedding (as for
positional encodings) and injected into a small U-Net.

Implement:

- a `SinusoidalTimeEmbedding` that maps an integer timestep to a vector
  of `dim` features, and
- a `SimpleUNet` with a two-level encoder/decoder and skip connections
  that adds the time embedding at the bottleneck.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        """t: (N,) integer timesteps -> (N, dim) embedding."""
        # TODO: Your solution here


class SimpleUNet(nn.Module):
    def __init__(self, in_channels=1, time_dim=64):
        super().__init__()
        # Hint: build a small encoder-decoder with two downsampling stages and skip connections.
        # The bottleneck should end with 64 channels so the projected time embedding can be added.
        # TODO: Your solution here

    def forward(self, x, t):
        # Hint: run encoder -> bottleneck -> decoder, add the projected time embedding in the
        # bottleneck, and concatenate the saved encoder activations before each decoder block.
        # TODO: Your solution here

In [ ]:
# TODO: Your solution here

### 2.3 Train the Diffusion Model \[3 points\]

Train the denoiser with the simple noise-prediction objective
$$\mathcal{L}(\theta) = \mathbb{E}_{x_0,\,\epsilon,\,t}\Bigl[\bigl\lVert \epsilon - \epsilon_\theta(x_t, t)\bigr\rVert^2\Bigr].$$

For each batch: draw a random timestep $t$ per image, form $x_t$ with
`forward_diffusion`, predict the noise, and minimize the MSE to the true
noise. Plot the loss curve.

In [ ]:
def train_diffusion(model, loader, epochs=15, lr=1e-3):
    # Hint: for each batch, sample a random timestep per image, form x_t with forward_diffusion,
    # predict the noise, then optimize the MSE to the true noise and track the epoch averages.
    # TODO: Your solution here


# TODO: Your solution here

### 2.4 Sample and Inspect \[2 points\]

Generate new digits with the stochastic reverse (ancestral) process
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Bigl(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t, t)\Bigr) + \sigma_t z,
\qquad z \sim \mathcal{N}(0, I),$$
adding noise $z$ at every step except the last ($t = 0$). Then answer
the short interpretation questions.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples=16):
    # Hint: start from Gaussian noise, iterate backward from T-1 to 0, apply the reverse
    # update at each step, and only add fresh noise while t_val > 0.
    # TODO: Your solution here

In [ ]:
# TODO: Your solution here

After inspecting the forward visualization and the generated grid,
answer:

1.  How does the number of diffusion steps $T$ affect sample quality and
    the cost of sampling?

2.  The generator never sees a likelihood term that forces it to cover
    every digit, yet the samples are usually diverse. Why does the
    noise-prediction objective avoid the mode collapse seen with GANs in
    Sheet 06?

## Task 3: Deterministic Sampling and Guidance \[4 points\]

**Learning objectives:**

- implement a deterministic few-step sampler and relate it to the
  probability-flow ODE, and
- reflect on why few-step sampling is possible and what guidance would
  add.

In Task 2 the reverse process was stochastic and used all $T$ steps. The
script explains that diffusion also has a *deterministic* counterpart,
the probability-flow ODE, which can be integrated over a much shorter
sub-sequence of timesteps. Here you implement a deterministic
**DDIM-style** update and study the quality/speed tradeoff. This is not
the same as training a separate flow-matching model, but it is a
concrete way to explore the deterministic-flow viewpoint from the
script.

### 3.1 Few-Step Deterministic Sampling \[2 points\]

Implement a deterministic sampler over an evenly spaced sub-sequence of
timesteps. For a step from $t$ to an earlier $s$, first predict the
clean image
$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}},$$
then re-noise it to level $s$ **without** adding fresh randomness:
$$x_s = \sqrt{\bar\alpha_s}\,\hat{x}_0 + \sqrt{1-\bar\alpha_s}\,\epsilon_\theta(x_t, t).$$

Reuse the **same** starting noise for every step count so the comparison
is fair, and show grids for several step counts.

In [ ]:
@torch.no_grad()
def ddim_sample(model, n_steps=50, x_init=None, n_samples=16):
    # Hint: optionally reuse a fixed starting noise, create a shorter timestep sequence,
    # compute x0_pred at each step, and move deterministically to the earlier step s.
    # TODO: Your solution here

In [ ]:
# TODO: Your solution here

### 3.2 Reflect on Few-Step Sampling and Guidance \[2 points\]

1.  With the same starting noise, how did the deterministic samples
    change as you reduced the number of steps from 200 to 10? Relate
    this to the script’s claim that the probability-flow ODE describes a
    deterministic trajectory. In what sense does this experiment only
    *suggest* why straighter paths (optimal transport, Schrödinger
    bridges) should help, rather than directly testing those ideas?

2.  Our model is unconditional, so it generates arbitrary digits.
    Suppose you wanted to control *which* digit is produced. Sketch how
    **classifier-free guidance** would be added: what change is needed
    during training, and how would the noise prediction be combined at
    sampling time? What does increasing the guidance weight trade off?